alphafold-3
boltz-2

data related to this binding event; homologous protein in data bank (training dataset)

three parts:
1) generating training datasets
2) building ML models
3) Testing predictions in the lab


**Training dataset from DEL screen (DEL-ML training set)**
- 28,778 molecules labeled as DEL-hit candidates
- 346,817 molecules labelled as non-DEL-hits

DEL is DNA tag attached to molecule and test if those bind to protein of interest
- DNA sequencing used to determine what is attached to protein of interest (to read DNA-tag attached to small molecules)
- this enables screening of billions of molecules
- catch is that because you have DNA-tag, this could impact binding (binding could be driven by tag; large data but noisy which can be challenge for ML models)

**Test Set - E-ASMS**
- 339,000 molecules 
- 138 hits confirmed by SPR (bioactive confirmed)

UMAP of chemical space of test and training set
- there is little overlap from this view between these two sets
- so training ml on kinda different data (a challenge in ML)
- they come from two different sources (which is causing this difference)


---


**Crosstalk Target protein**

Identify WDR91 ligands
- DEL training set generated by Hitgen publicily available from Aircheck

---

# Chemical Data and Structure

## Raw data to a representation

The chemical space of drug-like (small organic molecules) is estimated to be 10^60 molecules

"Machine learning (ML) is a field of study in artificial intelligence concernved with the development and study of **statistical algorithms** that can **learn from data and generalise to unseen data, and thus perform tasks without explicit instructions**"


Want to pass a function onto molecule and see if it binds or not
- Could use image but better way is to change molecules into numbers

**Molecules -> Numerical features**
1) Descriptors/fingerprints (e.g. Morgan, RDKit)
2) SMILES strings
3) Graph representations
...etc

**Converting Molecules to Numbers**



**How can you make a good representation for it?**

access of properties - 

# June 3, 2026 - Building supervised model

### TOC

1. Model
2. Training
- a) Train-test split
- b) loss function
- c) optimization
3.

### Training
a) data splitting

keep some data for testing model generalization

This mut be done in the first step to avoid _data leakage_

Data leakage is information from outside the training dataset that influences the model; leading to overly optimistic performance estimates 

#### What is a good split?
To measure generalization, we need to have training and test data are **independently and identically distributed** (i.i.d)
- Independent: each sample carries new information and doesn't just repeat what others already encode
- Identically distributed: the train and test data come from the same probability law

Most math in ML relied on i.i.d


### How can we make an i.i.d split?

Random splitting (most common way): randomly split so the train and test set represent the same distribution (e.g. 80% train: 20% test)
- While this is common, may not be the best for your case:
    - imbalance data (stratified sampling)
    - structured data
        - time series
        - grouped/clustered (group split)
        - time dependent data
    - very small datasets () 

Stratified sampling is good for imbalanced data
- stratified split perserves class proportions in both sets (1% in test is also 1% in train)

### Loss function
Want to define how well the model is performing
- done using Loss function (or cost function) which measures how bad model is on training dataset

Want loss function to be low

### Optimization
by minimizing the loss function, we update model parameters to fit the training dataset well
- iterative process

## Testing
After training, finding best model parameters, need to test with test set
- Different metrics you can use

- Mean absolute error (MAE)
- Max error (Max)
- Mean absolute Percentage error (MAPE)
- Mean squared error (MSE)


Can make linear models more advanced with kernel learning (adding in KNN with linear regression)
- smooth (smoother than KNN regression)
- global approx -> kernel weight are trainable
- highly flexible but not scalable (not good for large datasets)

Used to be popular with material science 

## Tree Based Models
- idea: recursively split feature space

**Ensemble learning** to smooth out noise you make have in data; by using multiple decision trees (randome forest)

**Ensemble with boosting: Gradient boosting**
- Boosting usually gives better performance
- Idea: building a strong model by _sequentially_ adding weak learners 
- each learner is trained to reduce the errors (residuals) of the current ensemble

Gradient boosting is model-agnostic
- the weak learned can be any differntiable function approximator
- GBDT: core idea of boosting
- XGBoost: optimized and regulatized GBDT
- LightGBM: GBDT optimized for speed and memory

Gradient boosting models are the strongest models on tabular data



## Neural Networks and Representation Learning



---

**Connection to uncertainty-based re-ranking.** Confidence weighting (`score = mean − λ·std`) is a form of uncertainty-aware ranking: the standard deviation across the per-fingerprint predictions serves as an uncertainty estimate, and we down-weight molecules where the fingerprints disagree. Empirically this was a marginal effect on our data — Random Forest preferred λ=0 (its bagged predictions are already variance-averaged, so the cross-fingerprint spread adds mostly noise), while XGBoost showed a small interior optimum at λ=0.5. We retain these settings but report the effect honestly as minor: uncertainty re-ranking did not meaningfully change performance here, likely because the dominant difficulty is the train/test distribution shift rather than per-molecule prediction variance.

**Adversarial validation and the question of test-aware splits.** The domain classifier above is an instance of *adversarial validation* — training a model to distinguish training (DEL) from test (E-ASMS) examples in order to quantify their similarity. Its ~100% accuracy is a strong adversarial-validation signal: the two distributions are almost perfectly separable.

This result also bears on whether we should use an *adversarial (test-aware) cross-validation split* — i.e., validating on the most test-like training examples so that offline scores better predict leaderboard performance. We did not adopt this, for two reasons grounded in our own results:

1. **Separability makes it infeasible.** Because the domains are ~100% separable, a test-similar validation set would draw from a thin, unrepresentative tail of the training data — collapsing to a small effective sample, exactly as importance weighting did (effective sample size ~3%). There is not enough test-like training chemistry to build a reliable test-aware validation fold.
2. **Grouped CV already transferred.** Empirically, the model ranking from our building-block–grouped cross-validation (XGBoost > Random Forest) held on the leaderboard, indicating that grouped CV — while in-domain — was a sufficiently faithful guide to relative model quality. An adversarial split was therefore unnecessary.

We thus rely on building-block–grouped cross-validation, using adversarial validation diagnostically (to characterize the shift) rather than to reweight or re-split the training data.